#インポート

In [0]:
%pip install openpyxl -q

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import glob

#ウィジェット

In [0]:
# ウィジェットの作成
dbutils.widgets.text("P_KIJYUN", "200512", "基準年月 (YYYYMM)")

# 値の取得と変数化
p_kijyun = dbutils.widgets.get("P_KIJYUN")
yyyy = p_kijyun[:4]
mm = int(p_kijyun[4:6]) 

#読み込み

##マスタ

In [0]:
# Excelファイルのパス定義
excel_path = "/Volumes/training_bs/common/input/master_file.xlsx"

###店マスタ

In [0]:
# シートを指定して読み込み
pdf_mise = pd.read_excel(excel_path, sheet_name="mise_mstr", dtype={"mise_cd": str})
df_mise = spark.createDataFrame(pdf_mise)

# 属性設定と型変換
df_mise = df_mise.select(
    F.col("MISE_CD").cast("string").alias("mise_cd"), 
    F.col("MISE_NM").cast("string").alias("mise_nm")
)
display(df_mise)

In [0]:
# カタログへ保存
spark.sql("DROP TABLE IF EXISTS user_uenishi.mise_mstr")
df_mise.write.mode("overwrite").saveAsTable("user_uenishi.mise_mstr")

# 保存確認
display(spark.read.table("user_uenishi.mise_mstr"))

###価格マスタ

In [0]:
# 価格マスタシートの読み込み
pdf_kakaku = pd.read_excel(excel_path, sheet_name="syohin_kakaku_mstr", dtype={"syohin_cd": str})
df_kakaku = spark.createDataFrame(pdf_kakaku)

display(df_kakaku.limit(10))

In [0]:
# 最新レコード抽出
window_spec = Window.partitionBy("SYOHIN_CD").orderBy(F.col("UPDATE").desc())

df_kakaku_latest = df_kakaku.withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")

# 型とフォーマットの整理
df_kakaku_latest = df_kakaku_latest.select(
    F.col("SYOHIN_CD").cast("string").alias("syohin_cd"),
    F.col("SYOHIN_TANKA").cast("double").alias("syohin_tanka"),
    F.try_to_date(F.col("UPDATE"), "yyyy-MM-dd HH:mm:ss").alias("update")
)
display(df_kakaku_latest.limit(10))

In [0]:
spark.sql("DROP TABLE IF EXISTS user_uenishi.syohin_kakaku_mstr")
df_kakaku_latest.write.mode("overwrite").saveAsTable("user_uenishi.syohin_kakaku_mstr")

display(spark.read.table("user_uenishi.syohin_kakaku_mstr"))

###商品名称マスタ

In [0]:
# 名称マスタシートの読み込み
pdf_name = pd.read_excel(excel_path, sheet_name="syohin_name_mstr1", dtype=str)
df_name = spark.createDataFrame(pdf_name)

df_name = df_name.select(
    F.col("SYOHIN_CD1").cast("string").alias("syohin_cd1"),
    F.col("SYOHIN_NM1").cast("string").alias("syohin_nm1")
)

spark.sql("DROP TABLE IF EXISTS user_uenishi.syohin_name_mstr1")
df_name.write.mode("overwrite").saveAsTable("user_uenishi.syohin_name_mstr1")
display(spark.read.table("user_uenishi.syohin_name_mstr1"))

##売上データ

In [0]:
# 入力ディレクトリパス
input_dir = "/Volumes/training_bs/common/input/"

for i in range(1, mm + 1):
    ym = f"{yyyy}{str(i).zfill(2)}"
    file_name = f"conv_uri_{ym}.csv"
    
    # CSV読み込み (カンマ区切り、ヘッダーあり)
    df_csv = spark.read.csv(f"{input_dir}{file_name}", header=True, encoding="cp932")
    
    # データ定義 (inputステートメント相当)
    df_uri_final = df_csv.select(
        F.to_date(F.col("売上年月日"), "yyyyMMdd").alias("YMD"),
        F.col("売上店舗コード").cast("string").alias("MISE_CD"),
        F.col("レシート番号").cast("string").alias("RENO"),
        F.col("商品コード").cast("string").alias("SYOHIN_CD"),
        F.col("売上個数").cast("double").alias("KOSU")
    )
    
    # テーブルとして保存
    table_name = f"user_uenishi.conv_uri_{ym}"
    df_uri_final.write.mode("overwrite").saveAsTable(table_name)
    
    print(f"Imported: {table_name}")